In [38]:
# =============================================================================
# RECOMMENDATION & BUSINESS LOGIC ENGINE
# Market Basket Analysis - School Supplies
# Deliverables: Top bundles, top rules, homepage ranking, frequently bought
#               together, cross-sell, promo generator, business insights
# =============================================================================

import pandas as pd
import numpy as np
import os
import warnings
from itertools import combinations
from collections import defaultdict

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', None)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')

# -----------------------------------------------------------------------------
# SECTION 1 - LOAD RULES
# Loads from automation engine outputs if available, otherwise runs fallback miner.
# -----------------------------------------------------------------------------

OUTPUTS_DIR  = 'outputs'
DATASET_A    = 'dataset_A.csv'
DATASET_B    = 'dataset_B.csv'
RULES_A_PATH = os.path.join(OUTPUTS_DIR, 'rules_dataset_A_standalone.csv')
RULES_B_PATH = os.path.join(OUTPUTS_DIR, 'rules_dataset_B_standalone.csv')

def fallback_mine_rules(csv_path, min_support=0.05, min_confidence=0.3):
    """Fallback pairwise miner used when automation engine outputs are missing."""
    df = pd.read_csv(csv_path)
    transactions = df.groupby('TransactionID')['Item'].apply(list).tolist()
    n = len(transactions)
    item_counts = defaultdict(int)
    pair_counts = defaultdict(int)
    for t in transactions:
        items = list(set(t))
        for item in items: item_counts[item] += 1
        for a, b in combinations(sorted(items), 2): pair_counts[(a, b)] += 1
    rules = []
    for (a, b), count in pair_counts.items():
        sup = count / n
        if sup < min_support: continue
        for ant, con in [(a, b), (b, a)]:
            conf = count / item_counts[ant]
            lift = conf / (item_counts[con] / n)
            score = sup * conf * lift
            if conf >= min_confidence:
                rules.append({'antecedent': ant, 'consequent': con,
                              'support': round(sup,4), 'confidence': round(conf,4),
                              'lift': round(lift,4), 'score': round(score,4)})
    return pd.DataFrame(rules).sort_values('score', ascending=False).reset_index(drop=True)

def clean_col(series):
    """Strip curly braces from frozenset strings e.g. '{Notebook}' -> 'Notebook'."""
    return series.astype(str).str.replace(r'[\{\}]', '', regex=True).str.strip()

if os.path.exists(RULES_A_PATH) and os.path.exists(RULES_B_PATH):
    rules_A = pd.read_csv(RULES_A_PATH)
    rules_B = pd.read_csv(RULES_B_PATH)
    print(f'Loaded from automation engine. A: {len(rules_A)} rules | B: {len(rules_B)} rules')
    SOURCE = 'automation_engine'
else:
    print('Automation outputs not found. Running fallback miner...')
    rules_A = fallback_mine_rules(DATASET_A)
    rules_B = fallback_mine_rules(DATASET_B)
    print(f'Fallback mining done. A: {len(rules_A)} rules | B: {len(rules_B)} rules')
    SOURCE = 'fallback_miner'

for df in [rules_A, rules_B]:
    df['antecedent'] = clean_col(df['antecedent'])
    df['consequent'] = clean_col(df['consequent'])

rules_A['dataset'] = 'A'
rules_B['dataset'] = 'B'
rules_all = pd.concat([rules_A, rules_B], ignore_index=True).sort_values('score', ascending=False)
print(f'Combined pool: {len(rules_all)} rules | Source: {SOURCE}')

# -----------------------------------------------------------------------------
# SECTION 2 - TOP RULES FORMATTING
# Displays top rules with plain-English interpretation.
# -----------------------------------------------------------------------------

def format_rules_table(rules_df, title, top_n=10):
    print('=' * 75)
    print(f'  {title}')
    print('=' * 75)
    top = rules_df.head(top_n).copy()
    top.index = range(1, len(top) + 1)
    display_cols = ['antecedent', 'consequent', 'support', 'confidence', 'lift', 'score']
    available = [c for c in display_cols if c in top.columns]
    print(top[available].to_string())
    print()
    print('  Plain-English Interpretation (Top 5):')
    print('  ' + '-' * 60)
    for _, row in top.head(5).iterrows():
        tag = 'Strong' if row['lift'] >= 1.2 else ('Moderate' if row['lift'] >= 1.0 else 'Weak')
        print(f'  [{tag}] Customers who buy [{row["antecedent"]}] -> {row["confidence"]*100:.1f}% also buy [{row["consequent"]}] (lift: {row["lift"]:.2f}x)')
    print()

format_rules_table(rules_A, 'TOP RULES - Dataset A (Elementary School Supplies)', top_n=10)
format_rules_table(rules_B, 'TOP RULES - Dataset B (High School / College Supplies)', top_n=10)

# -----------------------------------------------------------------------------
# SECTION 3 - TOP BUNDLES DISPLAY
# Identifies best product bundles based on rule score.
# -----------------------------------------------------------------------------

def get_top_bundles(rules_df, dataset_label='', top_n=5):
    seen, bundles = set(), []
    for _, row in rules_df.iterrows():
        pair = tuple(sorted([row['antecedent'], row['consequent']]))
        if pair not in seen:
            seen.add(pair)
            bundles.append({'Bundle': f"{pair[0]}  +  {pair[1]}",
                            'Support': row['support'], 'Confidence': row['confidence'],
                            'Lift': row['lift'], 'Score': row['score']})
        if len(bundles) == top_n: break
    df_out = pd.DataFrame(bundles)
    df_out.index = range(1, len(df_out) + 1)
    print('=' * 65)
    print(f'  TOP {top_n} PRODUCT BUNDLES - Dataset {dataset_label}')
    print('=' * 65)
    print(df_out.to_string())
    print()
    return df_out

bundles_A = get_top_bundles(rules_A, dataset_label='A (Elementary)', top_n=5)
bundles_B = get_top_bundles(rules_B, dataset_label='B (High School / College)', top_n=5)

# -----------------------------------------------------------------------------
# SECTION 4 - HOMEPAGE RANKING
# Ranks items by total rule score - highest ranked items should be featured first.
# -----------------------------------------------------------------------------

def homepage_ranking(rules_df, dataset_label='', top_n=10):
    item_scores = defaultdict(float)
    item_rule_count = defaultdict(int)
    for _, row in rules_df.iterrows():
        for col in ['antecedent', 'consequent']:
            item_scores[row[col]] += row['score']
            item_rule_count[row[col]] += 1
    ranking = pd.DataFrame([
        {'Rank': i+1, 'Item': item, 'Total Score': round(score,4), 'Rule Appearances': item_rule_count[item]}
        for i, (item, score) in enumerate(sorted(item_scores.items(), key=lambda x: x[1], reverse=True)[:top_n])
    ]).set_index('Rank')
    print('=' * 60)
    print(f'  HOMEPAGE ITEM RANKING - Dataset {dataset_label}')
    print('  Higher score = stronger association hub = feature first')
    print('=' * 60)
    print(ranking.to_string())
    print()
    return ranking

rank_A = homepage_ranking(rules_A, dataset_label='A (Elementary)', top_n=10)
rank_B = homepage_ranking(rules_B, dataset_label='B (High School / College)', top_n=10)

# -----------------------------------------------------------------------------
# SECTION 5 - FREQUENTLY BOUGHT TOGETHER
# Given an item, returns items most commonly purchased alongside it.
# -----------------------------------------------------------------------------

def frequently_bought_together(item, rules_df, top_n=5):
    matches = rules_df[rules_df['antecedent'].str.lower() == item.lower()].copy()
    if matches.empty:
        matches = rules_df[rules_df['consequent'].str.lower() == item.lower()].copy()
        if matches.empty:
            print(f'  [{item}] not found in any rules.')
            return
        matches = matches.rename(columns={'antecedent': 'consequent', 'consequent': 'antecedent'})
    matches = matches.sort_values('confidence', ascending=False).head(top_n)
    print(f'  Frequently Bought With: [{item}]')
    print('  ' + '-' * 50)
    for rank, (_, row) in enumerate(matches.iterrows(), 1):
        tag = '[Strong]' if row['lift'] >= 1.2 else '[Moderate]'
        print(f'  {rank}. {tag} {row["consequent"]:30s}  confidence: {row["confidence"]*100:.1f}%   lift: {row["lift"]:.2f}x')
    print()

print('=' * 60)
print('  FREQUENTLY BOUGHT TOGETHER')
print('=' * 60)
print()
for item in ['Notebook', 'Pencil', 'Eraser']:
    frequently_bought_together(item, rules_A, top_n=5)
for item in ['Ballpen', 'Calculator', 'Binder']:
    frequently_bought_together(item, rules_B, top_n=5)

# -----------------------------------------------------------------------------
# SECTION 6 - CROSS-SELL SUGGESTIONS
# Given a cart, suggests additional items based on association rules.
# -----------------------------------------------------------------------------

def cross_sell(cart_items, rules_df, top_n=5, dataset_label=''):
    cart_lower = [i.lower() for i in cart_items]
    suggestions = []
    has_score = 'score' in rules_df.columns
    for _, row in rules_df.iterrows():
        if row['antecedent'].lower() in cart_lower and row['consequent'].lower() not in cart_lower:
            entry = {'Suggested Item': row['consequent'], 'Because You Have': row['antecedent'],
                     'Confidence': row['confidence'], 'Lift': row['lift']}
            if has_score: entry['Score'] = row['score']
            suggestions.append(entry)
    if not suggestions:
        print(f'  No suggestions for cart: {cart_items}')
        return
    sug_df = pd.DataFrame(suggestions)
    sort_col = 'Score' if 'Score' in sug_df.columns else 'Lift'
    sug_df = sug_df.sort_values(sort_col, ascending=False).drop_duplicates(subset='Suggested Item').head(top_n).reset_index(drop=True)
    sug_df.index = range(1, len(sug_df) + 1)
    label = f' [{dataset_label}]' if dataset_label else ''
    print(f'  Cross-Sell Suggestions{label}')
    print(f'  Current Cart: {" | ".join(cart_items)}')
    print('  ' + '-' * 65)
    print(sug_df.to_string())
    print()

print('=' * 70)
print('  CROSS-SELL ENGINE')
print('=' * 70)
print()
cross_sell(['Pencil', 'Notebook'], rules_A, top_n=5, dataset_label='Elementary')
cross_sell(['Crayon Set', 'Scissors'], rules_A, top_n=5, dataset_label='Elementary')
cross_sell(['Ballpen', 'Bond Paper'], rules_B, top_n=5, dataset_label='HS/College')
cross_sell(['Calculator', 'Graphing Paper'], rules_B, top_n=5, dataset_label='HS/College')

# -----------------------------------------------------------------------------
# SECTION 7 - PROMO GENERATOR
# Auto-generates bundle discount promos from top rules with tiered discounts.
# -----------------------------------------------------------------------------

PRICE_LIST = {
    'Notebook': 35, 'Pencil': 10, 'Eraser': 8, 'Sharpener': 12,
    'Pencil Case': 55, 'Ruler': 20, 'Colored Pencils': 75, 'Crayon Set': 90,
    'Watercolor Set': 120, 'Glue Stick': 25, 'Scissors': 40, 'Folder': 18,
    'Highlighter': 30, 'Correction Tape': 28, 'Index Cards': 22,
    'Intermediate Pad': 32, 'Paper Clips': 15, 'Sticky Notes': 28,
    'Compass': 45, 'Protractor': 25,
    'Ballpen': 12, 'Bond Paper': 55, 'Gel Pen': 35, 'Mechanical Pencil': 65,
    'Calculator': 280, 'Graphing Paper': 28, 'Binder': 95, 'Clearbook': 75,
    'Yellow Pad': 38, 'Stapler': 85, 'Staple Wire': 18, 'Puncher': 70,
    'Scotch Tape': 22, 'Double-Sided Tape': 30, 'Permanent Marker': 40,
    'Whiteboard Marker': 45, 'Ballpen Ink Refill': 20, 'Art Paper': 15,
    'Geometry Set': 120, 'Plastic Envelope': 25
}

def assign_discount(lift, confidence):
    if lift >= 1.3 and confidence >= 0.45: return 15, 'Hot Deal'
    elif lift >= 1.1 and confidence >= 0.35: return 10, 'Bundle Saver'
    else: return 5, 'Starter Bundle'

def promo_generator(rules_df, dataset_label='', top_n=5):
    seen, promos = set(), []
    for _, row in rules_df.iterrows():
        ant, con = row['antecedent'], row['consequent']
        pair = tuple(sorted([ant, con]))
        if pair in seen: continue
        seen.add(pair)
        total = PRICE_LIST.get(ant, 50) + PRICE_LIST.get(con, 50)
        disc_pct, tag = assign_discount(row['lift'], row['confidence'])
        savings = round(total * disc_pct / 100, 2)
        promos.append({'Tag': tag, 'Bundle': f'{ant} + {con}',
                       'Regular (P)': total, 'Discount': f'{disc_pct}%',
                       'Save (P)': savings, 'Promo (P)': round(total - savings, 2),
                       'Lift': row['lift']})
        if len(promos) == top_n: break
    promo_df = pd.DataFrame(promos)
    promo_df.index = range(1, len(promo_df) + 1)
    print('=' * 80)
    print(f'  PROMO GENERATOR - Dataset {dataset_label}')
    print('=' * 80)
    print(promo_df.to_string())
    print()
    return promo_df

promos_A = promo_generator(rules_A, dataset_label='A (Elementary)', top_n=5)
promos_B = promo_generator(rules_B, dataset_label='B (High School / College)', top_n=5)

# -----------------------------------------------------------------------------
# SECTION 8 - BUSINESS INSIGHTS
# Automated summary of key findings comparing both datasets.
# -----------------------------------------------------------------------------

def business_insights(rules_A, rules_B):
    print('=' * 75)
    print('  BUSINESS INSIGHTS REPORT - School Supplies Kiosk')
    print('=' * 75)

    print(f'\n1. Rule Volume')
    print(f'   Dataset A (Elementary)         : {len(rules_A)} rules')
    print(f'   Dataset B (High School/College) : {len(rules_B)} rules')
    richer = 'B' if len(rules_B) > len(rules_A) else 'A'
    print(f'   -> Dataset {richer} has richer association patterns.')

    avg_lift_A = rules_A['lift'].mean() if not rules_A.empty else 0
    avg_lift_B = rules_B['lift'].mean() if not rules_B.empty else 0
    print(f'\n2. Average Lift')
    print(f'   Dataset A: {avg_lift_A:.4f}  |  Dataset B: {avg_lift_B:.4f}')
    stronger = 'A' if avg_lift_A > avg_lift_B else 'B'
    print(f'   -> Dataset {stronger} has stronger purchasing associations on average.')

    print(f'\n3. Strongest Rule Per Dataset')
    for label, rules_df in [('A', rules_A), ('B', rules_B)]:
        if not rules_df.empty:
            r = rules_df.iloc[0]
            print(f'   Dataset {label}: [{r["antecedent"]}] -> [{r["consequent"]}]  confidence: {r["confidence"]*100:.1f}%  lift: {r["lift"]:.2f}x  score: {r["score"]:.4f}')

    print(f'\n4. Most Connected Items (Hub Products)')
    for label, rules_df in [('A', rules_A), ('B', rules_B)]:
        counts = defaultdict(int)
        for _, row in rules_df.iterrows():
            counts[row['antecedent']] += 1
            counts[row['consequent']] += 1
        top3 = sorted(counts.items(), key=lambda x: x[1], reverse=True)[:3]
        print(f'   Dataset {label}: ' + ', '.join([f'{item} ({n} rules)' for item, n in top3]))
    print('   -> Hub products should be featured prominently on the shelf and homepage.')

    print(f'\n5. Cross-Dataset Overlap')
    pairs_A = set(zip(rules_A['antecedent'], rules_A['consequent']))
    pairs_B = set(zip(rules_B['antecedent'], rules_B['consequent']))
    shared = pairs_A & pairs_B
    jaccard = len(shared) / len(pairs_A | pairs_B) if (pairs_A | pairs_B) else 0
    print(f'   Jaccard Similarity: {jaccard:.4f}  |  Shared: {len(shared)}  |  Only A: {len(pairs_A - pairs_B)}  |  Only B: {len(pairs_B - pairs_A)}')
    verdict = 'LOW - maintain separate recommendation engines per segment.' if jaccard < 0.2 else ('MODERATE - some patterns transfer but segment-specific rules matter.' if jaccard < 0.5 else 'HIGH - a unified model is viable.')
    print(f'   -> Overlap is {verdict}')

    print(f'\n6. Actionable Recommendations')
    recs = [
        'Bundle top 3 rule pairs into labeled combo deals at the counter.',
        'Place hub products at eye level on the shelf.',
        'Use cross-sell engine at checkout to suggest 1-2 items per basket.',
        'Run Hot Deal bundles during back-to-school season.',
        'Segment promos: simpler bundles for elementary, broader for HS/College.',
        'Refresh rules monthly as buyer patterns shift across semesters.',
    ]
    for i, r in enumerate(recs, 1): print(f'   {i}. {r}')
    print()
    print('=' * 75)
    print('  Business Insights Report Complete')
    print('=' * 75)

business_insights(rules_A, rules_B)

# -----------------------------------------------------------------------------
# SECTION 9 - SAVE OUTPUTS
# Exports all outputs as CSV files to rec_outputs/
# -----------------------------------------------------------------------------

os.makedirs('rec_outputs', exist_ok=True)
bundles_A.to_csv('rec_outputs/top_bundles_A.csv')
bundles_B.to_csv('rec_outputs/top_bundles_B.csv')
rank_A.to_csv('rec_outputs/homepage_ranking_A.csv')
rank_B.to_csv('rec_outputs/homepage_ranking_B.csv')
promos_A.to_csv('rec_outputs/promos_A.csv', index=False)
promos_B.to_csv('rec_outputs/promos_B.csv', index=False)
rules_A.head(20).to_csv('rec_outputs/top_rules_A.csv', index=False)
rules_B.head(20).to_csv('rec_outputs/top_rules_B.csv', index=False)

print('All outputs saved to: rec_outputs/')
for f in sorted(os.listdir('rec_outputs')):
    print(f'   rec_outputs/{f}')
print('\nRecommendation & Business Logic Engine - Complete')


Loaded from automation engine. A: 211 rules | B: 206 rules
Combined pool: 417 rules | Source: automation_engine
  TOP RULES - Dataset A (Elementary School Supplies)
             antecedent   consequent  support  confidence   lift  score
1       Correction Tape  Pencil Case   0.0825      0.4195 1.0896 0.8034
2           Highlighter  Pencil Case   0.1042      0.3968 1.0307 0.7736
3              Notebook       Pencil   0.1075      0.3525 1.0654 0.7587
4        Watercolor Set     Notebook   0.0458      0.3667 1.2022 0.7540
5              Notebook  Pencil Case   0.1167      0.3825 0.9935 0.7520
6                Pencil     Notebook   0.1075      0.3249 1.0654 0.7251
7   Eraser, Highlighter  Pencil Case   0.0333      0.4301 1.1172 0.7246
8       Colored Pencils  Pencil Case   0.0808      0.3927 1.0200 0.7034
9            Crayon Set     Notebook   0.0617      0.3442 1.1285 0.6971
10          Index Cards       Pencil   0.0500      0.3704 1.1195 0.6933

  Plain-English Interpretation (Top 5):
  